# 02 - Perbandingan SVM dengan dan tanpa mRMR

Notebook ini menjaga data uji tetap terpisah. TF-IDF, mRMR, dan tuning SVM hanya di-fit pada data latih melalui cross-validation. Pipeline scikit-learn ini berjalan di CPU; `GridSearchCV(n_jobs=-1)` menggunakan seluruh core CPU yang tersedia.

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.feature_selection import MRMRSelector
from app.preprocessor import clean_text

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'comments_labeled.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
REPORT_DIR = PROJECT_ROOT / 'reports'
ARTIFACT_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 10
FAST_MODE = False  # True untuk uji alur cepat sebelum eksperimen final
PRIMARY_METRIC = 'f1_macro'
LABEL_ORDER = ['negatif', 'netral', 'positif']

## Memuat dan memvalidasi dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'{DATA_PATH} belum tersedia. Selesaikan anotasi dari notebook 01 terlebih dahulu.'
    )

df = pd.read_csv(DATA_PATH)
required_columns = {'text', 'label_final', 'split'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Kolom wajib tidak tersedia: {sorted(missing_columns)}')

df = df.dropna(subset=['text', 'label_final']).copy()
df['text'] = df['text'].astype(str).str.strip()
df['label_final'] = df['label_final'].astype(str).str.casefold().str.strip()
df['split'] = df['split'].astype(str).str.casefold().str.strip()
label_aliases = {
    'negative': 'negatif', 'neg': 'negatif', '-1': 'negatif',
    'neutral': 'netral', 'neut': 'netral', '0': 'netral',
    'positive': 'positif', 'pos': 'positif', '1': 'positif',
}
df['label_final'] = df['label_final'].replace(label_aliases)
invalid_labels = sorted(set(df['label_final']) - set(LABEL_ORDER))
if invalid_labels:
    raise ValueError(f'Label tidak valid: {invalid_labels}')
invalid_splits = sorted(set(df['split']) - {'train', 'test'})
if invalid_splits:
    raise ValueError(f'Split tidak valid: {invalid_splits}')
if df['label_final'].value_counts().min() < 2:
    raise ValueError('Setiap kelas membutuhkan minimal dua sampel untuk train-test split.')

print('Jumlah data valid:', len(df))
display(df[['text', 'label_final']].head())

In [ ]:
distribution = df['label_final'].value_counts().reindex(LABEL_ORDER, fill_value=0)
display(pd.DataFrame({
    'jumlah': distribution,
    'persentase': (distribution / distribution.sum() * 100).round(2),
}))

sns.set_theme(style='whitegrid')
ax = distribution.plot.bar(color=['#c2413b', '#64748b', '#15803d'])
ax.set(title='Distribusi Label Final', xlabel='Kelas', ylabel='Jumlah')
plt.xticks(rotation=0)
plt.show()

## Split data sebelum TF-IDF dan mRMR

Kolom `split` dibuat sebelumnya oleh `scripts/prepare_svm_train_test_split.py` dengan stratifikasi label dan `random_state=42`. Data uji yang sudah dikunci tidak digunakan untuk Grid Search, TF-IDF fitting, atau seleksi fitur mRMR.

In [ ]:
train_df = df.loc[df['split'].eq('train')].copy()
test_df = df.loc[df['split'].eq('test')].copy()
if train_df.empty or test_df.empty:
    raise ValueError('Split train dan test harus sama-sama berisi data.')
X_train, y_train = train_df['text'], train_df['label_final']
X_test, y_test = test_df['text'], test_df['label_final']

minimum_train_class = y_train.value_counts().min()
effective_folds = min(CV_FOLDS, int(minimum_train_class))
if effective_folds < 2:
    raise ValueError('Data latih setiap kelas belum cukup untuk cross-validation.')
cv = StratifiedKFold(
    n_splits=effective_folds, shuffle=True, random_state=RANDOM_STATE
)

print(f'Train: {len(X_train)} | Test: {len(X_test)} | CV folds: {effective_folds}')
display(pd.crosstab(index=y_train, columns='train'))
display(pd.crosstab(index=y_test, columns='test'))

## Pipeline dan ruang pencarian parameter

In [ ]:
tfidf = TfidfVectorizer(
    preprocessor=clean_text,
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    max_features=10_000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

baseline_pipeline = Pipeline([
    ('tfidf', tfidf),
    ('svm', SVC(class_weight='balanced', decision_function_shape='ovr')),
])

mrmr_pipeline = Pipeline([
    ('tfidf', tfidf),
    ('mrmr', MRMRSelector(random_state=RANDOM_STATE)),
    ('svm', SVC(class_weight='balanced', decision_function_shape='ovr')),
])

if FAST_MODE:
    svm_grid = [{'svm__kernel': ['linear'], 'svm__C': [1]}]
    mrmr_k = [100]
else:
    svm_grid = [
        {'svm__kernel': ['linear'], 'svm__C': [0.1, 1, 10]},
        {'svm__kernel': ['rbf'], 'svm__C': [0.1, 1, 10], 'svm__gamma': ['scale', 0.01, 0.1]},
    ]
    mrmr_k = [100, 200, 300, 500]

baseline_grid = svm_grid
mrmr_grid = [{**params, 'mrmr__k': mrmr_k} for params in svm_grid]

## Grid Search pada data latih

Eksperimen final dapat memerlukan waktu lama, terutama mRMR K=500 dengan 10-fold CV. Gunakan `FAST_MODE=True` untuk memastikan seluruh alur berjalan sebelum eksperimen final.

In [ ]:
def train_search(name: str, pipeline: Pipeline, parameters: list[dict]):
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameters,
        scoring=PRIMARY_METRIC,
        cv=cv,
        n_jobs=-1,
        verbose=2,
        refit=True,
        return_train_score=False,
    )
    started = time.perf_counter()
    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - started
    print(f'{name}: {elapsed:.2f} detik')
    print('Best CV Macro-F1:', round(search.best_score_, 4))
    print('Best parameters:', search.best_params_)
    return search, elapsed

baseline_search, baseline_train_seconds = train_search(
    'SVM tanpa mRMR', baseline_pipeline, baseline_grid
)
mrmr_search, mrmr_train_seconds = train_search(
    'SVM + mRMR', mrmr_pipeline, mrmr_grid
)

## Evaluasi final pada test set

In [ ]:
def evaluate(name: str, search: GridSearchCV, train_seconds: float):
    started = time.perf_counter()
    prediction = search.predict(X_test)
    predict_seconds = time.perf_counter() - started
    report = classification_report(
        y_test, prediction, labels=LABEL_ORDER, output_dict=True, zero_division=0
    )
    result = {
        'model': name,
        'cv_macro_f1': search.best_score_,
        'test_accuracy': accuracy_score(y_test, prediction),
        'test_macro_f1': f1_score(y_test, prediction, average='macro'),
        'test_weighted_f1': f1_score(y_test, prediction, average='weighted'),
        'train_seconds': train_seconds,
        'predict_seconds': predict_seconds,
        'best_params': search.best_params_,
        'classification_report': report,
        'prediction': prediction,
    }
    return result

baseline_result = evaluate('SVM tanpa mRMR', baseline_search, baseline_train_seconds)
mrmr_result = evaluate('SVM + mRMR', mrmr_search, mrmr_train_seconds)
results = [baseline_result, mrmr_result]

comparison = pd.DataFrame([{
    key: result[key]
    for key in [
        'model', 'cv_macro_f1', 'test_accuracy', 'test_macro_f1',
        'test_weighted_f1', 'train_seconds', 'predict_seconds'
    ]
} for result in results]).set_index('model')
display(comparison.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, result in zip(axes, results):
    matrix = confusion_matrix(y_test, result['prediction'], labels=LABEL_ORDER)
    ConfusionMatrixDisplay(matrix, display_labels=LABEL_ORDER).plot(
        ax=axis, cmap='Blues', colorbar=False
    )
    axis.set_title(result['model'])
plt.tight_layout()
plt.show()

## Fitur mRMR terpilih dan analisis perbedaan prediksi

In [ ]:
best_mrmr = mrmr_search.best_estimator_
feature_names = best_mrmr.named_steps['tfidf'].get_feature_names_out()
selected_indices = best_mrmr.named_steps['mrmr'].get_support(indices=True)
selected_features = pd.DataFrame({
    'feature': feature_names[selected_indices],
    'relevance_mi': best_mrmr.named_steps['mrmr'].relevance_scores_[selected_indices],
}).sort_values('relevance_mi', ascending=False)
display(selected_features.head(30))

prediction_comparison = pd.DataFrame({
    'text': X_test.to_numpy(),
    'actual': y_test.to_numpy(),
    'baseline_prediction': baseline_result['prediction'],
    'mrmr_prediction': mrmr_result['prediction'],
})
prediction_comparison['models_differ'] = (
    prediction_comparison['baseline_prediction'] != prediction_comparison['mrmr_prediction']
)
display(prediction_comparison[prediction_comparison['models_differ']].head(20))

## Simpan model dan laporan

In [ ]:
joblib.dump(mrmr_search.best_estimator_, ARTIFACT_DIR / 'model_a_mrmr.joblib')
joblib.dump(baseline_search.best_estimator_, ARTIFACT_DIR / 'model_b_baseline.joblib')
selected_features.to_csv(REPORT_DIR / 'selected_features_mrmr.csv', index=False)
prediction_comparison.to_csv(REPORT_DIR / 'test_predictions_comparison.csv', index=False)
comparison.to_csv(REPORT_DIR / 'model_comparison.csv')

serializable_report = []
for result in results:
    serializable_report.append({
        key: value
        for key, value in result.items()
        if key not in {'prediction'}
    })
with (REPORT_DIR / 'experiment_details.json').open('w', encoding='utf-8') as handle:
    json.dump(serializable_report, handle, indent=2, ensure_ascii=False, default=float)

print('Model dan laporan tersimpan.')
print('Artifacts:', ARTIFACT_DIR)
print('Reports:', REPORT_DIR)

## Checklist sebelum hasil digunakan dalam skripsi

- [ ] Label test set sudah diverifikasi manusia.
- [ ] Test set tidak digunakan untuk memilih K atau parameter SVM.
- [ ] Kedua skenario menggunakan split dan parameter pencarian yang sama.
- [ ] Hasil CV dilaporkan sebagai rata-rata dan standar deviasi.
- [ ] Macro-F1, metrik per kelas, confusion matrix, dan efisiensi dilaporkan.
- [ ] Contoh salah klasifikasi dianalisis.
- [ ] Notebook final dijalankan ulang dari kernel bersih tanpa error.